# 4. Autoencoder inference and latent-space images

A trained autoencoder exposes `encode`, `decode`, and whole-image `transform`. Latent values are saved as a new imzML/ibd pair with the original spatial coordinates and metadata, but with the spectral axis replaced by latent-component indices. This makes latent data readable and sliceable through the same image-like API.

In [2]:
import os 
from pathlib import Path

# seting global dir
cwd=Path.cwd()
if cwd.name == "tutorials":
    # os.chdir(cwd.parent.parent) 
    os.chdir(cwd.parent.parent.parent) 
os.getcwd()

'/home/maxi7524/repositories/MSIAutoEncoderWrapper'

In [4]:
from pathlib import Path
import numpy as np
import torch
from msi_autoencoder_wrapper.core.wrapper import MSIAutoEncoderWrapper
from msi_autoencoder_wrapper.models.datasets.strategies.pixel_dataset import PixelDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
wrapper = MSIAutoEncoderWrapper("data/tutorial_workspace", device=device)
image_path = Path("data/tutorial_workspace/imgs/example.imzML").resolve()
wrapper.context_manager.set_reader("PyImzMLReader", str(image_path))
wrapper.context_manager.set_binner("LinearBinning", str(image_path), bin_step=0.1)
wrapper.context_manager.set_inverse_binner(
    "TopPeaksInverseBinner", str(image_path), max_bins=1500, window_size=3
)
wrapper.workspace.set_active_image(str(image_path))

2026-07-19 16:33:24,540 | INFO     | msi_autoencoder_wrapper.core.wrapper:70 | MSIAutoEncoderWrapper: Anchoring processing device state: cuda
2026-07-19 16:33:24,542 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:46 | Enforcing automatic module discovery for reader and binner registries.
2026-07-19 16:33:24,543 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 2 implementation module(s) in package 'msi_autoencoder_wrapper.readers.strategies'.
2026-07-19 16:33:24,544 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 1 implementation module(s) in package 'msi_autoencoder_wrapper.binners.binners_strategies'.
2026-07-19 16:33:24,546 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 1 implementation module(s) in package 'msi_autoencoder_wrapper.binners.inverse_strategies'.
2026-07-19 16:33:24,547 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 10 implementati

## Loaded model versus image-local model

`models_manager.model_functionality` belongs to the one currently loaded model. `active_context.local_model_functionality` belongs to the selected image. Binding is optional because it retains a second long-lived reference to the model. Bind when the image must keep using this model after another model is loaded.

In [13]:
# Run after the example model bundle is installed in the workspace.
wrapper.models_manager.load_model(
    img_name="example",
    model_name="example-autoencoder",
    bind_to_local_context=True,
)
loaded_autoencoder = wrapper.models_manager.model_functionality
local_autoencoder = wrapper.active_context.local_model_functionality
assert loaded_autoencoder is local_autoencoder

2026-07-19 16:38:13,193 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 20 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures'.
2026-07-19 16:38:13,194 | INFO     | msi_autoencoder_wrapper.models.model_loader:65 | Reconstructing loaded model family 'autoencoder'.
2026-07-19 16:38:13,194 | INFO     | msi_autoencoder_wrapper.models.architectures.architectures_manager:125 | Initializing multi-component sub-graph resolution phase for model family: autoencoder
2026-07-19 16:38:13,195 | INFO     | msi_autoencoder_wrapper.models.architectures.architectures_manager:144 | Instantiating standard component sub-module: Category='encoder' using Strategy='CNNEncoder'.
2026-07-19 16:38:13,198 | INFO     | msi_autoencoder_wrapper.models.architectures.architectures_manager:144 | Instantiating standard component sub-module: Category='decoder' using Strategy='CNNDecoder'.
2026-07-19 16:38:13,201 | INFO     | msi_autoencoder_wrapper.models.architectu

Loading a replacement with `bind_to_local_context=False` changes only the manager interface. `wrapper.active_context.autoencoder` prefers the local binding, while `wrapper.models_manager.autoencoder` always addresses the currently loaded model. This distinction is important when comparing models without losing an image's established transform.

In [ ]:
wrapper.models_manager.load_model(
    img_name="example",
    model_name="comparison-autoencoder",
    bind_to_local_context=False,
)
assert wrapper.active_context.autoencoder is local_autoencoder
assert wrapper.models_manager.autoencoder is not local_autoencoder

## Encode and decode explicit batches

`encode` expects binned `[batch, bins]` data. `decode(..., grid_xs=True)` returns the regular binned grid. The default `grid_xs=False` applies the configured inverse binner and returns sparse `(m/z, intensity)` pairs. Both methods switch to evaluation mode and disable gradient tracking.

In [9]:
xs, ys = wrapper.active_context.reader[0]
grid = wrapper.active_context.binner(xs=xs, ys=ys)
# autoencoder = wrapper.active_context.autoencoder
# latent_vector = autoencoder.encode(grid[None, :])
# reconstructed_grid = autoencoder.decode(latent_vector, grid_xs=True)
# reconstructed_sparse = autoencoder.decode(latent_vector, grid_xs=False)
# print(latent_vector.shape, reconstructed_grid.shape)

## Transform and save a complete latent image

Whole-image transform needs an active dataset associated with the current image. A loaded model can be used without its original image, but transformation cannot start until an image dataset is supplied. `save_latent` writes both `.imzML` and `.ibd`, then can activate the new latent reader without unloading the original reader.

In [ ]:
wrapper.workspace.set_active_image('example')
wrapper.active_dataset = PixelDataset(
    active_context=wrapper.active_context,
    source="image",
)
latent_matrix = wrapper.active_context.autoencoder.transform(
    {"batch_size": 128, "num_workers": 0, "pin_memory": device == "cuda"}
)
latent_path = wrapper.active_context.save_latent(
    output_path="data/tutorial_workspace/models/example/example-autoencoder/latent/example.latent.imzML",
    loader_config={"batch_size": 128, "num_workers": 0},
    activate=True,
)

#TODO - dostajemy bląd - NameErro: name 'AutoencoderContextInterface' is not defined pomimo że jest podłazcony - trzeba to przejrezć i zobaczyć co nei działa, chciałbym takżeżębyś miw ytlumaczyl jake te mocdlee (autoencodry są podpijena )

2026-07-19 16:38:16,021 | INFO     | msi_autoencoder_wrapper.core.mixins.workspace.proxies.getters_and_setters_proxy:75 | Active image context mapped by index key: example


NameError: name 'AutoencoderContextInterface' is not defined

## Work on image and latent spaces independently

`data_source` controls default image-like operations, while an explicit `source=` avoids hidden switching. The original and latent readers are not both loaded on demand: each is resolved only when requested, and the original may be absent in a latent-only workflow. Coordinate order and slice semantics are identical in both spaces.

In [ ]:
# wrapper.active_context.set_data_source("latent")
# latent_spectrum = wrapper.active_context.get_spectrum(0, source="latent")
# latent_region = wrapper.active_context.get_region(
#     slice(10, 20), slice(30, 40), source="latent"
# )
# original_region = wrapper.active_context.get_region(
#     slice(10, 20), slice(30, 40), source="image"
# )
# latent_dataset = PixelDataset(active_context=wrapper.active_context, source="latent")

A latent-only consumer can create an empty workspace and call `active_context.load_latent(path)`. It can slice latent data and build `PixelDataset(source="latent")` without opening the original image. Decoding with a loaded autoencoder also works with `grid_xs=True`; sparse decoding additionally needs the matching inverse binner.

## Monitor memory

Run the commands available on the current platform. Linux commands also work in WSL. `nvidia-smi` is supplied with the NVIDIA driver; it is not useful for Apple MPS. macOS provides `vm_stat` and `memory_pressure`. `psutil` is optional (`pip install psutil`) because the library has a standard-library RAM fallback.

In [ ]:
# Linux / WSL:
# !free -h
# !df -h .
# NVIDIA GPU:
# !nvidia-smi
# macOS:
# !vm_stat
# !memory_pressure
# !df -h .

In [ ]:
if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print({"free_vram": free_bytes, "total_vram": total_bytes})
    print(torch.cuda.memory_summary(abbreviated=True))

For predictable memory use, begin with the estimator from Tutorial 3, use a smaller transform batch, keep `num_workers=0` on constrained machines, avoid binding models locally unless persistence is needed, and process spatial slices rather than materializing every spectrum. `unload_latent()` releases the latent reader; `models_manager.unload_model()` releases only the loaded-model reference. A local binding intentionally keeps its model alive. `torch.cuda.empty_cache()` releases unused cached CUDA blocks, not tensors that still have references.

In [ ]:
# wrapper.active_context.unload_latent()
# wrapper.models_manager.unload_model()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()